In [1]:
import numpy as np
import pickle
import time
from datetime import timedelta
from scipy.optimize import linear_sum_assignment, linprog
import os


In [2]:
def load_avalanche_data(filepath, min_time_length=1, max_time_length=None):
    """
    Load avalanche data from pickle file
    
    Args:
        filepath: path to the pickle file
        min_time_length: minimum temporal length required (default: 1)
        max_time_length: maximum temporal length allowed (default: None, no upper limit)
    
    Returns:
        avalanche_arrays: list of avalanche arrays with temporal length in [min_time_length, max_time_length]
    """
    with open(filepath, 'rb') as f:
        avalanches = pickle.load(f)
    
    print(f"Loaded {len(avalanches)} participants")
    
    # Extract arrays from ALL participants  
    avalanche_arrays = []
    filtered_count_too_short = 0
    filtered_count_too_long = 0
    
    for participant_idx, participant_avalanches in enumerate(avalanches):
        for avalanche_idx, avalanche in enumerate(participant_avalanches):
            # avalanche is already a numpy array, not a list
            if hasattr(avalanche, 'shape') and avalanche.ndim == 2:
                temporal_length = avalanche.shape[1]
                
                # Check temporal length constraints
                if temporal_length < min_time_length:
                    filtered_count_too_short += 1
                elif max_time_length is not None and temporal_length > max_time_length:
                    filtered_count_too_long += 1
                else:
                    avalanche_arrays.append(avalanche)
                    if participant_idx == 0 and len(avalanche_arrays) <= 3:  # Debug first few
                        print(f"  Added avalanche {avalanche_idx}: shape {avalanche.shape}")
    
    print(f"Total avalanches extracted: {len(avalanche_arrays)}")
    print(f"Filtered out {filtered_count_too_short} avalanches with temporal length < {min_time_length}")
    if max_time_length is not None:
        print(f"Filtered out {filtered_count_too_long} avalanches with temporal length > {max_time_length}")
    
    return avalanche_arrays

def create_distance_matrix(filepath='/home/novicky/clean_code/processed_avalanches_250Hz/eeg_ground_metric.pkl'):
    """
    Load physical distance matrix between electrodes from pickle file and normalize by mean
    """
    try:
        with open(filepath, 'rb') as f:
            K = pickle.load(f)
        
        print(f"Loaded distance matrix K from {filepath}")
        print(f"K shape: {K.shape}")
        print(f"K range (original): [{K.min():.3f}, {K.max():.3f}]")
        print(f"K diagonal (should be zeros): {np.diag(K)[:5]}...")
        
        # Normalize by mean distance (better for alignment-based matching)
        mean_distance = K[K > 0].mean()  # Exclude zeros (self-distances)
        if mean_distance > 0:
            K_normalized = K / mean_distance
            print(f"Normalized by mean distance: {mean_distance:.3f}")
            print(f"K range (normalized): [{K_normalized.min():.3f}, {K_normalized.max():.3f}]")
            print(f"K mean (after normalization): {K_normalized[K_normalized > 0].mean():.3f}")  # Should be ~1.0
            print(f"K diagonal (after normalization): {np.diag(K_normalized)[:5]}...")
            return K_normalized
        else:
            print("Warning: mean distance is 0, returning original K")
            return K
    
    except FileNotFoundError:
        return None
    
    except Exception as e:
        return None

def spatial_alignment_distance(x_pattern, y_pattern, K):
    """
    Calculate optimal transport distance with penalties for:
    1. Completely unmatched polarities (positive/negative mismatch)
    2. Cardinality imbalance within matched polarities
    
    Returns normalized cost per electrode involved in the comparison.
    """
    
    pos_x_idx = np.where(x_pattern == 1)[0]   
    neg_x_idx = np.where(x_pattern == -1)[0]  
    pos_y_idx = np.where(y_pattern == 1)[0]   
    neg_y_idx = np.where(y_pattern == -1)[0]  
    
    # Calculate mean distance for penalties
    mean_distance = np.mean(K[K > 0])
    
    total_cost = 0
    
    # Handle positive activities
    if len(pos_x_idx) > 0 and len(pos_y_idx) > 0:
        # Both have positives - compute transport with cardinality penalty
        total_cost += solve_optimal_assignment(pos_x_idx, pos_y_idx, K)
    elif len(pos_x_idx) > 0 or len(pos_y_idx) > 0:
        # One has positives, other doesn't - "vanishing" penalty
        unmatched = len(pos_x_idx) + len(pos_y_idx)
        total_cost += unmatched * mean_distance
    
    # Handle negative activities
    if len(neg_x_idx) > 0 and len(neg_y_idx) > 0:
        # Both have negatives - compute transport with cardinality penalty
        total_cost += solve_optimal_assignment(neg_x_idx, neg_y_idx, K)
    elif len(neg_x_idx) > 0 or len(neg_y_idx) > 0:
        # One has negatives, other doesn't - "vanishing" penalty
        unmatched = len(neg_x_idx) + len(neg_y_idx)
        total_cost += unmatched * mean_distance
    
    # Normalize by total electrodes involved in this timestep comparison
    total_electrodes_involved = np.sum(x_pattern != 0) + np.sum(y_pattern != 0)
    
    if total_electrodes_involved > 0:
        normalized_cost = total_cost / total_electrodes_involved
    else:
        normalized_cost = 0  # Both patterns are silent
    
    return normalized_cost


def solve_optimal_assignment(source_idx, target_idx, K):
    """
    Solve optimal assignment using Hungarian algorithm with minimum-based dummy nodes.
    Unmatched nodes pay their minimum possible distance.
    
    Args:
        source_idx: indices of source electrodes
        target_idx: indices of target electrodes
        K: distance matrix
    
    Returns:
        total_cost: optimal assignment cost including penalties for unmatched nodes
    """
    n_source = len(source_idx)
    n_target = len(target_idx)
    
    if n_source == 0 or n_target == 0:
        return 0
    
    # Extract base cost matrix
    cost_matrix = K[np.ix_(source_idx, target_idx)]
    
    if n_source == n_target:
        # Balanced case - standard Hungarian
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        return cost_matrix[row_ind, col_ind].sum()
    
    elif n_source < n_target:
        
        # Create augmented matrix (n_target × n_target)
        augmented_matrix = np.zeros((n_target, n_target))
        
        # Fill real sources (first n_source rows)
        augmented_matrix[:n_source, :] = cost_matrix
        
        # Fill dummy sources (rows n_source onward)
        # Each column (target) gets its minimum distance from any real source
        for j in range(n_target):
            min_dist_to_target_j = cost_matrix[:, j].min()
            augmented_matrix[n_source:, j] = min_dist_to_target_j
        
        # Solve with Hungarian
        row_ind, col_ind = linear_sum_assignment(augmented_matrix)
        return augmented_matrix[row_ind, col_ind].sum()
    
    else:  # n_source > n_target
        
        # Create augmented matrix (n_source × n_source)
        augmented_matrix = np.zeros((n_source, n_source))
        
        # Fill real targets (first n_target columns)
        augmented_matrix[:, :n_target] = cost_matrix
        
        # Fill dummy targets (columns n_target onward)
        # Each row (source) gets its minimum distance to any real target
        for i in range(n_source):
            min_dist_from_source_i = cost_matrix[i, :].min()
            augmented_matrix[i, n_target:] = min_dist_from_source_i
        
        # Solve with Hungarian
        row_ind, col_ind = linear_sum_assignment(augmented_matrix)
        return augmented_matrix[row_ind, col_ind].sum()

def create_temporal_similarity_matrix(avalanche_x, avalanche_y, K, show_progress=False):
    """
    Create similarity matrix between time steps of two avalanches
    """
    M = avalanche_x.shape[1]
    N = avalanche_y.shape[1]
    
    similarity_matrix = np.zeros((M, N))
    total_pairs = M * N
    
    if show_progress and total_pairs > 3000:
        print(f"    Computing {M}×{N} = {total_pairs} time step pairs...", end="")
        start_time = time.time()
    
    for i in range(M):
        for j in range(N):
            x_pattern = avalanche_x[:, i]  
            y_pattern = avalanche_y[:, j]  
            
            # Just compute the distance - penalties handle everything
            distance = spatial_alignment_distance(x_pattern, y_pattern, K)
            similarity_matrix[i, j] = distance
            
            # Progress for large matrices
            if show_progress and total_pairs > 3000:
                current = i * N + j + 1
                if current % max(1, total_pairs // 10) == 0:
                    elapsed = time.time() - start_time
                    progress = current / total_pairs
                    eta = elapsed / progress - elapsed if progress > 0 else 0
                    # print(f" {progress:.0%} (ETA: {eta:.1f}s)", end="")
    
    if show_progress and total_pairs > 3000:
        elapsed = time.time() - start_time
        print(f" Done! ({elapsed:.2f}s)")
    
    return similarity_matrix

def backtrack_greedy(dtw_matrix, similarity_matrix, end_i, end_j, penalty):
    """
    Simple greedy backtrack - check in preference order, take first match.
    """
    if np.isinf(dtw_matrix[end_i, end_j]):
        return []
    
    path = []
    i, j = end_i, end_j
    
    while j > 0:
        path.append((i-1, j-1))
        cell_cost = similarity_matrix[i-1, j-1]
        
        # Try diagonal (preferred)
        if i > 0 and j > 0 and not np.isinf(dtw_matrix[i-1, j-1]):
            if np.isclose(dtw_matrix[i, j], cell_cost + dtw_matrix[i-1, j-1], atol=1e-9):
                i, j = i-1, j-1
                continue
        
        # Try up (second choice)
        if i > 0 and not np.isinf(dtw_matrix[i-1, j]):
            if np.isclose(dtw_matrix[i, j], cell_cost + dtw_matrix[i-1, j] + penalty, atol=1e-9):
                i, j = i-1, j
                continue
        
        # Try left (last resort)
        if j > 0 and not np.isinf(dtw_matrix[i, j-1]):
            if np.isclose(dtw_matrix[i, j], cell_cost + dtw_matrix[i, j-1] + 2 * penalty, atol=1e-9):
                i, j = i, j-1
                continue
        
        # No valid move found
        break
    
    return list(reversed(path))

def dtw_on_similarity_matrix(similarity_matrix):
    """
    Apply subsequence DTW on similarity matrix to find optimal temporal alignment.
    Allows starting anywhere on the left edge and ending anywhere on the right edge.
    Automatically ensures X (columns) is shorter than Y (rows) by transposing if needed.
    Uses adaptive penalty to reduce tie-induced path explosion.
    
    Args:
        similarity_matrix: (M, N) matrix (will be transposed if N > M)
    
    Returns:
        min_cost: minimum DTW distance
        paths: list of all optimal paths (each path is a list of (i,j) tuples)
        was_transposed: bool indicating if matrix was transposed
    """
    M, N = similarity_matrix.shape
    
    # Check if we need to transpose
    was_transposed = False
    if N > M:
        similarity_matrix = similarity_matrix.T
        M, N = similarity_matrix.shape
        was_transposed = True
        # print(f"      Transposed similarity matrix: now {M}×{N} (Y×X)")
    
    # Calculate adaptive penalty based on data statistics
    finite_costs = similarity_matrix[np.isfinite(similarity_matrix)]
    if len(finite_costs) > 0 and np.any(finite_costs > 0):
        min_nonzero_cost = np.min(finite_costs[finite_costs > 0])
        penalty = 0.001 * min_nonzero_cost  # 0.1% of minimum non-zero cost
        # print(f"      Adaptive penalty: {penalty:.6f} (0.1% of min cost {min_nonzero_cost:.3f})")
    else:
        penalty = 0.0
        # print(f"      No penalty (all costs are zero or inf)")
    
    # Initialize DTW matrix
    dtw_matrix = np.full((M + 1, N + 1), np.inf)

    # Calculate minimum required span
    min_required_span = N # The length of the shorter sequence

    # Only allow starting from rows that can achieve minimum span
    # If we start at row start_i, the maximum span we can achieve is (M - start_i)
    # We need: M - start_i >= min_required_span
    # So: start_i <= M - min_required_span
    max_start_row = int(M - min_required_span)

    # Allow starting from rows 0 to max_start_row
    for start_i in range(max_start_row + 1):
        dtw_matrix[start_i, 0] = 0
    
    # Fill DTW matrix with penalties for non-diagonal moves
    for i in range(1, M + 1):
        for j in range(1, N + 1):
            cost = similarity_matrix[i-1, j-1]
            
            # Apply penalties to non-diagonal moves
            candidates = [
                dtw_matrix[i-1, j-1],           # diagonal - no penalty
                dtw_matrix[i-1, j] + penalty,   # up (insertion) - small penalty
                dtw_matrix[i, j-1] + 2*penalty    # left (deletion) - 2x small penalty
            ]
            
            finite_candidates = [c for c in candidates if not np.isinf(c)]
            
            if len(finite_candidates) > 0: # This should not happen due to initialization
                dtw_matrix[i, j] = cost + min(finite_candidates)
            else:
                dtw_matrix[i, j] = np.inf

    
    # Valid ending positions sorted by cost
    valid_endings = [(dtw_matrix[end_i, N], end_i) 
                    for end_i in range(N, M + 1) 
                    if not np.isinf(dtw_matrix[end_i, N])]
    valid_endings.sort()  # Sort by cost

    # Try each ending position until we find a valid path
    for cost, end_i in valid_endings:
        # Backtrack greedily following minimum cost
        path = backtrack_greedy(dtw_matrix, similarity_matrix, end_i, N, penalty)
        
        if len(path) > 0:
            # Check span requirement
            start_row = path[0][0]
            end_row = path[-1][0]
            span = end_row - start_row + 1
            
            if span >= min_required_span:
                # Found valid path!
                min_cost = cost
                
                # Transpose path back if needed
                if was_transposed:
                    path = [(j, i) for (i, j) in path]
                
                return min_cost, [path], was_transposed

    # No valid path found from any ending
    return np.inf, [], was_transposed

In [3]:
def compute_avalanche_similarity_matrix_subset(
    avalanche_arrays, 
    K, 
    indices_A, 
    indices_B, 
    fill_symmetric=True,
    compute_upper_triangle_only=False
):
    """
    Compute similarity matrix for a subset of avalanche pairs (block computation).
    
    Args:
        avalanche_arrays: Full list of all avalanche arrays
        K: Physical distance matrix between electrodes
        indices_A: List/array of avalanche indices to use as rows
        indices_B: List/array of avalanche indices to use as columns
        fill_symmetric: If True, also compute transpose block (B vs A)
        compute_upper_triangle_only: If True, only compute upper triangle (for diagonal blocks)
    
    Returns:
        result_dict: Dictionary containing:
            - 'similarity_blocks': List of computed similarity blocks
            - 'path_blocks': List of computed path blocks
            - 'total_avalanches': Total number of avalanches in dataset
            - 'computed_blocks': List of (row_start, row_end, col_start, col_end) tuples
    """
    indices_A = list(indices_A)
    indices_B = list(indices_B)
    
    n_total = len(avalanche_arrays)
    n_rows = len(indices_A)
    n_cols = len(indices_B)
    
    # Check for diagonal overlap
    indices_A_set = set(indices_A)
    indices_B_set = set(indices_B)
    has_diagonal = len(indices_A_set & indices_B_set) > 0
    is_diagonal_block = (indices_A == indices_B)
    
    print(f"\n=== AVALANCHE SIMILARITY COMPUTATION (SUBSET) ===")
    print(f"Total avalanches in dataset: {n_total}")
    print(f"Block A (rows): {len(indices_A)} avalanches (indices {min(indices_A)} to {max(indices_A)})")
    print(f"Block B (cols): {len(indices_B)} avalanches (indices {min(indices_B)} to {max(indices_B)})")
    print(f"Diagonal block: {is_diagonal_block}")
    print(f"Diagonal overlap: {has_diagonal}")
    print(f"Fill symmetric: {fill_symmetric}")
    print(f"Compute upper triangle only: {compute_upper_triangle_only}")
    
    # Calculate actual pairs to compute
    if compute_upper_triangle_only and is_diagonal_block:
        total_pairs = n_rows * (n_rows + 1) // 2  # Upper triangle including diagonal
        print(f"Pairs to compute: {total_pairs} (upper triangle only)")
    else:
        total_pairs = n_rows * n_cols
        print(f"Pairs to compute: {total_pairs} (full block)")
    
    # Initialize storage for this block
    similarity_matrix_block = np.full((n_rows, n_cols), np.nan)
    path_matrix_block = [[None for _ in range(n_cols)] for _ in range(n_rows)]
    
    # Start timing
    overall_start = time.time()
    completed_pairs = 0
    
    print(f"\nStarting computation...")
    print("=" * 80)
    
    for local_i, global_i in enumerate(indices_A):
        row_start_time = time.time()
        
        # Progress report every 20 rows
        if local_i > 0 and local_i % 20 == 0:
            overall_elapsed = time.time() - overall_start
            avg_row_time = overall_elapsed / local_i
            remaining_rows = n_rows - local_i
            eta = avg_row_time * remaining_rows
            total_time = overall_elapsed + eta
            percent_done = (local_i / n_rows) * 100
            print(f"{percent_done:.1f}% | Running: {str(timedelta(seconds=int(overall_elapsed)))} | Total time to finish: {str(timedelta(seconds=int(total_time)))}")
        
        # Determine column range based on upper triangle setting
        if compute_upper_triangle_only and is_diagonal_block:
            col_start = local_i  # Start from diagonal
        else:
            col_start = 0  # Start from first column
        
        for local_j in range(col_start, n_cols):
            global_j = indices_B[local_j]
            
            # Handle diagonal case (i == j)
            if global_i == global_j:
                similarity_matrix_block[local_i, local_j] = 0.0
                path_matrix_block[local_i][local_j] = [[(t, t) for t in range(avalanche_arrays[global_i].shape[1])]]
                completed_pairs += 1
                continue
            
            # Regular computation
            avalanche_i_signed = avalanche_arrays[global_i]
            avalanche_j_signed = avalanche_arrays[global_j]
            
            # Create temporal similarity matrix
            temp_sim_matrix = create_temporal_similarity_matrix(
                avalanche_i_signed, avalanche_j_signed, K, show_progress=False
            )
            
            # Apply subsequence DTW
            dtw_distance, paths, was_transposed = dtw_on_similarity_matrix(temp_sim_matrix)
            
            # Store results
            similarity_matrix_block[local_i, local_j] = dtw_distance
            path_matrix_block[local_i][local_j] = paths
            
            completed_pairs += 1
            
            # If computing upper triangle only, mirror to lower triangle
            if compute_upper_triangle_only and is_diagonal_block and local_i != local_j:
                similarity_matrix_block[local_j, local_i] = dtw_distance
                # Transpose the paths for the symmetric entry
                transposed_paths = [[(j, i) for (i, j) in path] for path in paths]
                path_matrix_block[local_j][local_i] = transposed_paths
    
    total_time = time.time() - overall_start
    avg_time_per_pair = total_time / max(1, completed_pairs)
    
    print(f"\n=== BLOCK COMPUTATION COMPLETED ===")
    print(f"Total time: {str(timedelta(seconds=int(total_time)))}")
    print(f"Pairs computed: {completed_pairs}")
    print(f"Average time per pair: {avg_time_per_pair:.2f}s")
    print(f"Block shape: {similarity_matrix_block.shape}")
    
    # Check for NaN values
    nan_count = np.sum(np.isnan(similarity_matrix_block))
    if nan_count > 0:
        print(f"WARNING: {nan_count} NaN values in similarity matrix (expected if not computing full block)")
    
    print(f"Distance range: [{np.nanmin(similarity_matrix_block):.3f}, {np.nanmax(similarity_matrix_block):.3f}]")
    
    # Prepare result dictionary
    result = {
        'similarity_blocks': [
            {
                'row_indices': indices_A,
                'col_indices': indices_B,
                'matrix': similarity_matrix_block
            }
        ],
        'path_blocks': [
            {
                'row_indices': indices_A,
                'col_indices': indices_B,
                'paths': path_matrix_block
            }
        ],
        'total_avalanches': n_total,
        'computed_blocks': [(min(indices_A), max(indices_A) + 1, min(indices_B), max(indices_B) + 1)]
    }
    
    # Add symmetric block if requested and blocks are different
    if fill_symmetric and indices_A != indices_B:
        print(f"\nComputing symmetric block (B vs A)...")
        
        # Transpose the similarity matrix
        similarity_matrix_block_T = similarity_matrix_block.T
        
        # Transpose paths (swap i,j in each path)
        path_matrix_block_T = [[None for _ in range(n_rows)] for _ in range(n_cols)]
        for local_i in range(n_rows):
            for local_j in range(n_cols):
                if path_matrix_block[local_i][local_j] is not None:
                    # Swap coordinates in paths
                    transposed_paths = [[(j, i) for (i, j) in path] for path in path_matrix_block[local_i][local_j]]
                    path_matrix_block_T[local_j][local_i] = transposed_paths
        
        result['similarity_blocks'].append({
            'row_indices': indices_B,
            'col_indices': indices_A,
            'matrix': similarity_matrix_block_T
        })
        
        result['path_blocks'].append({
            'row_indices': indices_B,
            'col_indices': indices_A,
            'paths': path_matrix_block_T
        })
        
        result['computed_blocks'].append((min(indices_B), max(indices_B) + 1, min(indices_A), max(indices_A) + 1))
        
        print(f"Symmetric block added: B×A = {len(indices_B)}×{len(indices_A)}")
    
    return result

In [ ]:
if __name__ == "__main__":
    
    # ==========================================
    # CHANGE THIS NUMBER FOR PARALLEL RUNS
    BLOCK_ID = 0  # Change to 0, 1, 2, 3, etc. for each parallel job
    # ==========================================
    
    print("=== PARALLEL AVALANCHE DTW SIMILARITY COMPUTATION ===")
    
    # Configuration
    chunk_size = 2044
    fill_symmetric = True
    output_dir = 'similarity_blocks_with_polarity'
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output directory: {output_dir}/")
    
    # Load ALL avalanches first (needed to determine total count and chunks)
    print("Loading all avalanche data to determine chunk structure...")
    all_avalanche_arrays = load_avalanche_data(
        '/home/novicky/clean_code/processed_avalanches_250Hz/filtered_avalanche_peaks_250hz_twopoles.pkl',
        min_time_length=5,
        max_time_length=None
    )
    # Reverse polarity of each avalanche in the dataset by adding sign-flipped versions
    all_avalanche_arrays.extend([-arr for arr in all_avalanche_arrays])
    
    n_total = len(all_avalanche_arrays)
    n_chunks = (n_total + chunk_size - 1) // chunk_size  # Ceiling division
    
    # Calculate ALL unique chunk pairs (including diagonal)
    # For n_chunks=3: (0,0), (1,0), (1,1), (2,0), (2,1), (2,2)
    chunk_pairs = []
    for i in range(n_chunks):
        for j in range(i + 1):  # j goes from 0 to i (inclusive)
            chunk_A_start = i * chunk_size
            chunk_A_end = min((i + 1) * chunk_size, n_total)
            chunk_B_start = j * chunk_size
            chunk_B_end = min((j + 1) * chunk_size, n_total)
            chunk_pairs.append((chunk_A_start, chunk_A_end, chunk_B_start, chunk_B_end))
    
    n_blocks = len(chunk_pairs)
    
    print(f"\n{'='*80}")
    print(f"CALCULATION")
    print(f"{'='*80}")
    print(f"Total avalanches: {n_total}")
    print(f"Chunk size: {chunk_size}")
    print(f"Number of chunks: {n_chunks} (ceiling({n_total}/{chunk_size}))")
    print(f"Number of blocks to compute: {n_blocks} (all unique pairs including diagonal)")
    print(f"\nYou need to run {n_blocks} parallel jobs with BLOCK_ID from 0 to {n_blocks - 1}")
    
    print(f"\nBlock assignments:")
    for idx, (a_start, a_end, b_start, b_end) in enumerate(chunk_pairs):
        n_pairs = (a_end - a_start) * (b_end - b_start)
        is_diagonal = (a_start == b_start)
        sym_note = " (diagonal)" if is_diagonal else " (+ symmetric)" if fill_symmetric else ""
        print(f"  BLOCK_ID={idx}: Chunk[{a_start}:{a_end}] vs Chunk[{b_start}:{b_end}] ({n_pairs:,} pairs{sym_note})")
    
    # Validate BLOCK_ID
    if BLOCK_ID < 0 or BLOCK_ID >= len(chunk_pairs):
        print(f"\nERROR: BLOCK_ID={BLOCK_ID} is invalid!")
        print(f"Valid range: 0 to {len(chunk_pairs) - 1}")
        exit(1)
    
    # Get this block's assignment
    a_start, a_end, b_start, b_end = chunk_pairs[BLOCK_ID]
    is_diagonal = (a_start == b_start)
    
    # Extract only the avalanches needed for this block
    print(f"\nExtracting avalanches for BLOCK_ID={BLOCK_ID}...")
    min_idx = min(a_start, b_start)
    max_idx = max(a_end, b_end)
    
    # Keep only needed avalanches and create index mapping
    needed_avalanches = all_avalanche_arrays[min_idx:max_idx]
    print(f"Kept {len(needed_avalanches)} avalanches (indices {min_idx} to {max_idx})")
    
    # Adjust indices to new local coordinates
    local_a_start = a_start - min_idx
    local_a_end = a_end - min_idx
    local_b_start = b_start - min_idx
    local_b_end = b_end - min_idx
    
    # Delete the full array to free memory
    del all_avalanche_arrays
    
    print(f"\n{'='*80}")
    print(f"COMPUTING BLOCK_ID={BLOCK_ID}")
    print(f"Global Range A: [{a_start}:{a_end}] -> Local Range A: [{local_a_start}:{local_a_end}]")
    print(f"Global Range B: [{b_start}:{b_end}] -> Local Range B: [{local_b_start}:{local_b_end}]")
    print(f"Diagonal block: {is_diagonal}")
    print(f"{'='*80}")
    
    # # Load distance matrix
    # with open('/home/novicky/clean_code/processed_avalanches_250Hz/ground_metric_K.pkl', 'rb') as f:
    #     K = pickle.load(f)
    # # Normalize by mean distance (better for alignment-based matching)
    # mean_distance = K[K > 0].mean()
    # K_normalized = K / mean_distance
    K = create_distance_matrix('/home/novicky/clean_code/processed_avalanches_250Hz/eeg_ground_metric.pkl')

    # Compute this specific block using local indices
    start_time = time.time()
    
    # For diagonal blocks, don't fill symmetric (it's the same block)
    use_fill_symmetric = fill_symmetric and not is_diagonal
    
    block_dict = compute_avalanche_similarity_matrix_subset(
        avalanche_arrays=needed_avalanches,
        K=K,
        indices_A=range(local_a_start, local_a_end),
        indices_B=range(local_b_start, local_b_end),
        fill_symmetric=use_fill_symmetric,
        compute_upper_triangle_only=is_diagonal
    )
    
    # Fix the indices back to global coordinates in the result
    for block in block_dict['similarity_blocks']:
        block['row_indices'] = [idx + min_idx for idx in block['row_indices']]
        block['col_indices'] = [idx + min_idx for idx in block['col_indices']]
    
    for block in block_dict['path_blocks']:
        block['row_indices'] = [idx + min_idx for idx in block['row_indices']]
        block['col_indices'] = [idx + min_idx for idx in block['col_indices']]
    
    # Update computed_blocks to global coordinates
    block_dict['computed_blocks'] = [(a_start, a_end, b_start, b_end)]
    if use_fill_symmetric:
        block_dict['computed_blocks'].append((b_start, b_end, a_start, a_end))
    
    # Update total_avalanches to reflect the full dataset size
    block_dict['total_avalanches'] = n_total
    
    total_time = time.time() - start_time
    
    # Save result with block ID in filename to the output directory
    diagonal_tag = "_DIAG" if is_diagonal else ""
    output_filename = os.path.join(
        output_dir, 
        f'block_{BLOCK_ID:03d}_A{a_start}-{a_end}_B{b_start}-{b_end}{diagonal_tag}.pkl'
    )
    with open(output_filename, 'wb') as f:
        pickle.dump(block_dict, f)
    
    print(f"\n{'='*80}")
    print(f"BLOCK {BLOCK_ID} COMPLETED")
    print(f"{'='*80}")
    print(f"Total time: {str(timedelta(seconds=int(total_time)))}")
    print(f"Saved to: {output_filename}")
    print(f"\nTo run other blocks in parallel, change BLOCK_ID to:")
    for idx in range(min(5, len(chunk_pairs))):  # Show first 5 examples
        if idx != BLOCK_ID:
            a_s, a_e, b_s, b_e = chunk_pairs[idx]
            print(f"  BLOCK_ID = {idx}  # Chunk[{a_s}:{a_e}] vs Chunk[{b_s}:{b_e}]")
    if len(chunk_pairs) > 5:
        print(f"  ... (up to BLOCK_ID = {len(chunk_pairs) - 1})")